# Install, Configure, Troubleshoot & DCA Prep

## What's covered

- **Installing Docker Engine on Linux** — the official package paths, the convenience script, and why you generally pick the former
- **Post-install** — adding your user to the `docker` group, enabling on boot, where the daemon's data lives
- **Rootless Docker** — the alternative install for unprivileged daemon (recap from notebook 08)
- **The daemon configuration** — `/etc/docker/daemon.json`, every setting worth knowing, reload semantics
- **Storage drivers** — `overlay2` (the default), what `btrfs`, `zfs`, `devicemapper` are for, when (rarely) you'd change
- **Logging drivers** — `json-file`, `journald`, `local`, `syslog`, cloud drivers
- **The runtime layer** — `containerd`, `runc`, alternative runtimes (`crun`, `kata`, `gVisor`)
- **Daemon troubleshooting** — `journalctl -u docker`, `docker info`, disk and resource diagnostics
- **The container troubleshooting playbook** — the seven common failure modes and how to diagnose each
- **Build troubleshooting** — cache misses, BuildKit diagnostics
- **Network troubleshooting** — `nicolaka/netshoot`, `iptables`, route inspection
- **The DCA exam itself** — domains, weighting, format, sample question style
- **Exam-prep tactics** — what to practise, what to memorise, what to look up at test time
- **Where to go next** — Kubernetes, CKAD/CKA, cloud-specific certs

By the end you should be able to install, configure, and troubleshoot a Docker Engine install on a fresh Linux box, and walk into the DCA exam knowing what every section is asking about.

## Installing Docker Engine on Linux

The right way to install Docker Engine for any kind of production use is from **Docker's official package repositories**, not from a distro's default repos (which often lag months behind) and not from the convenience script (which is fine for throwaway VMs but skips integrity checks).

### Ubuntu / Debian

```bash
# 1. Install prereqs
sudo apt-get update
sudo apt-get install -y ca-certificates curl

# 2. Add Docker's official GPG key
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg \
  -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# 3. Add the repo
echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] \
  https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo $VERSION_CODENAME) stable" \
  | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

# 4. Install
sudo apt-get update
sudo apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
```

Five packages:

- **`docker-ce`** — the daemon (`dockerd`).
- **`docker-ce-cli`** — the `docker` CLI.
- **`containerd.io`** — the container runtime.
- **`docker-buildx-plugin`** — BuildKit-based builder (notebook 02).
- **`docker-compose-plugin`** — Compose v2 (notebook 06).

### RHEL / Fedora / CentOS / Rocky / Alma

```bash
sudo dnf install -y dnf-plugins-core
sudo dnf config-manager --add-repo https://download.docker.com/linux/centos/docker-ce.repo
sudo dnf install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
```

(Adjust the repo URL for your distro family — `fedora` vs `centos`.)

### The convenience script

```bash
curl -fsSL https://get.docker.com | sh
```

One line. Pulls and runs the official install script. Fine for ephemeral VMs or learning. **Don't use in production** — no signature pinning, no version control, makes future upgrades opaque.

## Post-install

A fresh install needs three or four small adjustments before it's pleasant to use.

### Run `docker` without `sudo`

By default, `/var/run/docker.sock` is owned by `root:docker`. To talk to it without `sudo`, add your user to the `docker` group:

```bash
sudo usermod -aG docker $USER
newgrp docker          # re-evaluate group membership in this shell
docker ps              # should work without sudo now
```

**Security caveat:** the `docker` group is effectively root on the host. A user in `docker` can `docker run -v /:/host alpine sh` and do anything. On shared hosts, prefer rootless Docker.

### Enable on boot

```bash
sudo systemctl enable --now docker
```

`enable` makes Docker start at boot; `--now` also starts it right now if it isn't already running.

### Verify

```bash
docker version           # client + server versions
docker info              # daemon configuration, storage driver, kernel, etc
docker run hello-world   # one full end-to-end test
```

### Where Docker's data lives

By default, everything is under `/var/lib/docker/`:

- `overlay2/` — image and container layers
- `volumes/` — named volumes
- `containers/<id>/` — container metadata and logs
- `image/`, `network/`, `plugins/`, `swarm/`, etc.

The daemon's own config lives at `/etc/docker/daemon.json` (covered next). The systemd unit is `/lib/systemd/system/docker.service`.

## Rootless Docker — the alternative install

Recap from notebook 08: rootless Docker runs the daemon as your unprivileged user, not as root. Different install path:

```bash
# As your normal user (NOT with sudo)
curl -fsSL https://get.docker.com/rootless | sh
# Follow the printed instructions to add ~/.bin to PATH and DOCKER_HOST to your env

systemctl --user enable docker
systemctl --user start docker
docker info   # should show "Security Options: rootless"
```

The rootless daemon uses `slirp4netns` for networking and stores its state under `~/.local/share/docker/`. Trade-offs covered in notebook 08 — slower networking, no `--privileged`, no `iptables` writes — but a much smaller attack surface.

## The daemon configuration — `/etc/docker/daemon.json`

`dockerd` reads `/etc/docker/daemon.json` at startup. The file is optional; with no config, defaults apply. The full schema is huge; here are the settings that matter in practice.

```json
{
  "data-root": "/mnt/docker-data",
  "storage-driver": "overlay2",

  "log-driver": "json-file",
  "log-opts": {
    "max-size": "10m",
    "max-file": "3"
  },

  "default-runtime": "runc",
  "runtimes": {
    "crun": { "path": "/usr/bin/crun" }
  },

  "registry-mirrors": ["https://my-cache.local"],
  "insecure-registries": ["registry.lab.internal:5000"],

  "userns-remap": "default",
  "no-new-privileges": true,
  "userland-proxy": false,
  "live-restore": true,

  "default-address-pools": [
    { "base": "172.30.0.0/16", "size": 24 }
  ],

  "metrics-addr": "127.0.0.1:9323",
  "experimental": false
}
```

What each block does:

- **`data-root`** — move Docker's data dir elsewhere. The default `/var/lib/docker` is often on the root partition; moving to a dedicated disk avoids filling the OS root.
- **`storage-driver`** — `overlay2` (default), `btrfs`, `zfs`, `devicemapper`. We discuss next.
- **`log-driver`** + **`log-opts`** — global default. Without `max-size`/`max-file`, json-file logs grow until the disk fills. Always set these.
- **`default-runtime`** + **`runtimes`** — `runc` is default. Add alternates (`crun`, `kata-runtime`, `gvisor`) and pick per-container with `docker run --runtime=...`.
- **`registry-mirrors`** — try this list first on pulls (the cache from notebook 07).
- **`insecure-registries`** — allow HTTP (no TLS) for these hosts. Lab use only.
- **`userns-remap`** — user-namespace remapping from notebook 08.
- **`userland-proxy: false`** — disable the docker-proxy fallback (notebook 05), iptables only.
- **`live-restore: true`** — daemon can restart without killing running containers. Useful during daemon upgrades.
- **`default-address-pools`** — pool that user-defined bridges allocate from. Avoid clashes with your corporate LAN subnets.
- **`metrics-addr`** — exposes Prometheus-format metrics for Docker itself.

After editing, reload:

```bash
sudo systemctl reload docker     # picks up some changes
sudo systemctl restart docker    # required for storage-driver, data-root, userns-remap
```

`docker info` always reflects what's actually loaded — verify after restart.

## Storage drivers

The storage driver decides how layers and the writable layer are stacked on disk.

| Driver | Backend | Notes |
|---|---|---|
| **`overlay2`** *(default)* | overlayfs kernel feature | Fast, stable, what almost everyone uses. The default since Docker 18. |
| **`btrfs`** | btrfs filesystem | Native snapshots, good for COW-heavy workloads. Requires `/var/lib/docker` on a btrfs subvolume. |
| **`zfs`** | ZFS filesystem | Similar pitch to btrfs. Hardly used on Linux outside specific shops. |
| **`devicemapper`** *(deprecated)* | LVM device-mapper | Legacy. Was the default on early RHEL. **Deprecated.** |
| **`vfs`** | Plain copies | No COW. Each layer is a full copy. Slow, huge disk use. Only for testing. |
| **`fuse-overlayfs`** | overlay via FUSE | Used by rootless Docker on kernels without unprivileged overlay support. |

For 99% of installs, **keep `overlay2`**. If you want to verify:

```bash
docker info | grep -i storage
# Storage Driver: overlay2
# Backing Filesystem: extfs
# Supports d_type: true
# Using metacopy: true
# Native Overlay Diff: true
# userxattr: false
```

Switching storage drivers is destructive — the daemon won't read another driver's data. Plan it for a maintenance window with full image/volume backups.

## Logging drivers

The default `json-file` driver writes each container's stdout/stderr to `/var/lib/docker/containers/<id>/<id>-json.log`. `docker logs` reads from there.

Other drivers ship logs elsewhere:

| Driver | Where logs go | `docker logs` works? |
|---|---|---|
| **`json-file`** *(default)* | Local file per container | Yes |
| **`local`** | Local file, compressed, faster | Yes |
| **`journald`** | systemd journal | Yes |
| **`syslog`** | Local or remote syslog | No |
| **`gelf`** | Graylog Extended Log Format (UDP/TCP) | No |
| **`fluentd`** | Fluentd collector | No |
| **`awslogs`** | AWS CloudWatch | No |
| **`gcplogs`** | Google Cloud Logging | No |
| **`splunk`** | Splunk HEC | No |
| **`none`** | Discarded | No |

Pick at three levels:

- **Daemon default** (`log-driver` in `daemon.json`).
- **Per container** (`docker run --log-driver=... --log-opt key=value`).
- **Per Compose service** (`logging:` block).

Most teams keep `json-file` with rotation on dev hosts, and switch to a centralised driver (`fluentd`, `gelf`, `awslogs`, `splunk`) on prod. If you switch the daemon default after containers exist, *existing containers don't migrate* — only newly-created ones use the new driver.

## The runtime layer — `containerd`, `runc`, alternatives

From notebook 01: the daemon doesn't actually run containers. `containerd` (the runtime supervisor) talks to `runc` (the low-level runtime) which does the namespace/cgroup/exec dance.

You usually don't touch this layer. Cases where you might:

- **`crun`** — a C-based reimplementation of `runc` that starts containers ~10x faster. Drop-in replacement at low risk; configure with `default-runtime: crun`.
- **`kata-runtime`** — runs each container inside a lightweight VM. Heavier than runc, but real hardware isolation. Used in multi-tenant clusters where the kernel-shared model isn't trusted enough.
- **`gvisor` / `runsc`** — userspace kernel sandbox. Implements much of the Linux syscall surface in Go, in userspace. Strong isolation, modest perf cost. Used by Google internally and by some PaaS providers.

To use them per-container after registering in `daemon.json`:

```bash
docker run --runtime=kata-runtime alpine sh
```

**Standalone containerd.** `containerd` can run without Docker. Kubernetes uses it directly via the CRI (Container Runtime Interface). Your Docker daemon and a Kubernetes node can share one `containerd` instance, which is how `docker` and `kubectl` can see overlapping resources on the same machine.

## Daemon troubleshooting

A small set of commands answers most "what's wrong with Docker?" questions.

```bash
docker info              # daemon configuration, all current settings
docker version           # client + server, useful for "is the daemon up?"
docker system df         # disk usage breakdown
docker system df -v      # per-image, per-container detail
docker system events     # live stream of every event the daemon sees
docker system prune      # clean stopped containers, unused images, networks
docker system prune -a   # also remove ALL unused images
docker system prune --volumes   # also wipe unused volumes — careful
```

### When the daemon won't start

```bash
sudo systemctl status docker      # is it actually running?
sudo journalctl -u docker -n 200  # last 200 lines of daemon log
sudo journalctl -u docker -f      # follow live
sudo dockerd --debug              # run in foreground with debug logging
```

Common causes:

- **Bad `daemon.json`.** Syntax error or unknown key. `sudo dockerd --validate` checks syntax.
- **Storage driver mismatch.** `/var/lib/docker` was created by one driver, you reconfigured to another. Either move it aside or revert the config.
- **Port already in use.** Another process is on 2375/2376/etc.
- **Permissions on `/var/lib/docker`.** Wrong ownership after a botched move.

### When the daemon is up but slow

```bash
docker system df    # is /var/lib/docker on a tiny disk?
top, iotop          # daemon process busy? disk I/O saturated?
docker events --since 5m   # stuck pulls or builds making noise?
```

### Disk filling up

```bash
docker system df              # how bad
du -sh /var/lib/docker/*      # which subdir is fattest
docker system prune -a        # nuclear option, often the right call
```

## The container troubleshooting playbook

The seven container failure modes you'll see, with the first commands to run.

### 1. "It won't start"

```bash
docker ps -a --filter name=myc      # what's its status?
docker logs myc                     # what did its main process say?
docker inspect myc | jq '.State'    # exit code, OOMKilled flag, started/finished times
```

Common causes: bad command/entrypoint, missing files, immediate crash, OOM (exit 137).

### 2. "It starts then exits immediately"

```bash
docker logs myc
docker run --rm -it IMAGE sh        # boot a shell instead, poke around
docker run --rm -it --entrypoint sh IMAGE   # bypass entrypoint
```

If you can't see why, run the container with the same flags but `sh` (or `bash`) as the command — gives you a shell to debug from.

### 3. "It's running but the app inside isn't healthy"

```bash
docker exec -it myc sh              # get a shell inside
docker top myc                      # what processes are running?
docker exec myc ps -ef              # same, from inside
docker exec myc netstat -tlnp       # what's listening?
docker stats --no-stream myc        # CPU, memory, I/O
```

### 4. "I can't reach it from outside"

```bash
docker port myc                     # what's published?
curl http://localhost:8080          # can the host reach it?
sudo iptables -t nat -L DOCKER -n   # is the DNAT rule there?
docker logs myc                     # is the app actually listening on 0.0.0.0?
```

Common cause: the app inside binds `127.0.0.1` instead of `0.0.0.0`. Inside the container, `127.0.0.1` is reachable only from inside — port publishing can't see it.

### 5. "Two containers can't talk to each other"

```bash
docker network ls                                     # which networks exist?
docker inspect myc -f '{{.NetworkSettings.Networks}}' # what is myc attached to?
docker inspect other -f '{{.NetworkSettings.Networks}}'
docker run --rm --network <net> nicolaka/netshoot ping other   # debug from the network
```

If they're on different user-defined networks, they cannot reach each other. If they're on the default bridge, they cannot resolve each other by name (notebook 05).

### 6. "It's slow"

```bash
docker stats myc                                       # live CPU/mem/I/O
docker inspect myc -f '{{.HostConfig.Memory}}, {{.HostConfig.NanoCpus}}'  # what limits?
docker logs --since 5m myc                             # recent app log
```

If `Memory > 0` and the app is paging, raise the limit. If `NanoCpus` is set and the app is CPU-bound, raise it.

### 7. "It died with status 137"

Exit code 137 = SIGKILL = `--memory` limit hit (OOMKilled) or someone `docker kill`'d it.

```bash
docker inspect myc -f '{{.State.OOMKilled}}'    # the smoking gun
dmesg | grep -i 'killed process'                # kernel-level OOM message on the host
```

Raise the memory limit, or fix the app's memory hog.

## Build troubleshooting

`docker build` has its own debugging path, mostly via BuildKit's plumbing.

```bash
# Build with full verbose output (no log line collapsing)
docker buildx build --progress=plain -t myimg .

# Print the resolved build context size
DOCKER_BUILDKIT=1 docker build --progress=plain --no-cache .

# Inspect what's in the build cache
docker buildx du --verbose

# Wipe the build cache (force rebuilds)
docker builder prune
docker buildx prune --all   # nuclear
```

### Common build problems

**Cache misses where you expect hits.** Usually a `COPY` whose source changed. `--progress=plain` shows you each step's cache state. Often: a stale `.dockerignore` is letting irrelevant files into the context.

**"unknown instruction" or "syntax" errors.** You're missing the BuildKit syntax pragma. Add `# syntax=docker/dockerfile:1.7` as the *first* line.

**"failed to compute cache key: ... not found"**. Source path doesn't exist in the context. Check `.dockerignore` and the path itself.

**Slow `COPY .` step.** Build context is huge. `du -sh .` to see size, `.dockerignore` to trim.

**Build hangs on `apt-get update`.** Daemon's DNS or network is broken. Check `docker run --rm alpine ping -c1 archive.ubuntu.com` from the same host.

## Network troubleshooting

When packets aren't flowing, `nicolaka/netshoot` is the universal Docker network debug image — it ships with `tcpdump`, `dig`, `nmap`, `ip`, `curl`, `mtr`, and dozens of others.

```bash
# Attach netshoot to a service's network and poke
docker run --rm -it --network app-net nicolaka/netshoot
# inside:
ping api
dig api
curl -v http://api/health
tcpdump -i any port 8080
```

Key commands inside netshoot or any debug container:

- **`ip a`** — interfaces and IPs (host-network or container-network depending on attach).
- **`ip route`** — routing table.
- **`dig <name>`** — DNS resolution (uses container's resolv.conf).
- **`curl -v <url>`** — HTTP-level verbose (TLS, redirects, timing).
- **`tcpdump -i any -nn 'port 8080'`** — see actual packets.
- **`nslookup <name>` / `getent hosts <name>`** — DNS in different forms.

### Inspecting the daemon's iptables rules

```bash
sudo iptables -t nat -L DOCKER -n -v   # NAT rules for port publishing
sudo iptables -L DOCKER-USER -n -v     # user-defined chain (hook for custom rules)
```

These are what the daemon manages. If a published port isn't reachable, the absence of the corresponding `DNAT` rule is the cause.

## The DCA exam itself

The **Docker Certified Associate** is administered by Mirantis (who acquired Docker Enterprise). The exam tests practical Docker knowledge across six domains.

### Format

- **55 multiple-choice / multiple-select** questions
- **90 minutes**
- **Online proctored**, taken at home or in-office
- **Passing score**: typically around 65-70% (Mirantis hasn't published an exact number)
- **Validity**: 2 years

### Domain weights (approximate)

| Domain | Weight | Topics |
|---|---|---|
| **Orchestration** | ~25% | Swarm, services, stacks, rolling updates, scaling, placement, secrets |
| **Image creation, management, registry** | ~20% | Dockerfile, multi-stage builds, tagging, registries, image-related security |
| **Installation and configuration** | ~15% | Installing Docker Engine, daemon.json, drivers, contexts |
| **Networking** | ~15% | Bridge, host, none, overlay, macvlan, publishing, troubleshooting |
| **Security** | ~15% | Non-root, capabilities, secrets, image signing, runtime sandboxing |
| **Storage and volumes** | ~10% | Volumes, bind mounts, drivers, plugins, mount syntax |

Each of those domain pillars is covered in this curriculum — notebooks 02 (images), 03 (runtime), 04 (storage), 05 (networking), 06 (compose, peripherally tested), 08 (security), 09 (orchestration), 10 (install/configure).

## Exam-prep tactics

### What to memorise vs look up

**Memorise** (the exam doesn't give you man pages):

- The shape of `docker run`, the most-used flags (`-d`, `-p`, `-v`, `--rm`, `-it`, `--restart`, `--name`, `--network`, `-u`, `--memory`, `--cpus`).
- Storage: volume vs bind mount vs tmpfs semantics; the on-mount-empty seeding rule.
- Networking: when containers can resolve each other by name, the routing-mesh model in Swarm.
- Dockerfile: every common instruction and its quirks (`CMD` vs `ENTRYPOINT`, `COPY` vs `ADD`, `ENV` vs `ARG`).
- Swarm: services vs tasks, replicated vs global, the rolling-update flags, `docker service` vs `docker stack`.
- Compose deploy block keys: `replicas`, `update_config`, `placement`, `restart_policy`, `resources`.

**Look up at test time** (you'd waste study cycles memorising):

- Exact `daemon.json` keys you don't use weekly.
- The full set of seccomp/AppArmor profile paths.
- Specific cloud-registry hostname schemes.

### What to practise (not just read)

Set up an actual Swarm cluster (multipass or three Lima VMs on a Mac):

```
docker swarm init / join
docker service create / scale / update / rollback
docker stack deploy from a stack.yaml
docker network create overlay
docker secret create / use
docker node update --availability drain / active
```

You should be able to do each from memory. Reading about them isn't enough.

### Sample-question style

The DCA leans on "given this scenario, what command…" or "which of these statements is true." A few stylised examples — work them and check against the right answer with this curriculum's content:

> **Q1.** You want to update a service named `api` to use image `myorg/api:1.4`, updating two replicas at a time with a 5-second delay, and rolling back automatically on failure. Which command does this?
>
> A. `docker service rollback api 1.4 -p 2 -d 5s`
> B. `docker service update --image myorg/api:1.4 --update-parallelism 2 --update-delay 5s --update-failure-action rollback api`
> C. `docker stack deploy --image myorg/api:1.4 --parallel 2`
> D. `docker container update api --image myorg/api:1.4`

> **Q2.** Which command prints all containers, including stopped ones, with their names and status, in a tabular format?
>
> A. `docker ps -a`
> B. `docker container ls --all --format 'table {{.Names}}\t{{.Status}}'`
> C. `docker inspect --status all`
> D. Both A and B are valid

The correct answers are **B** and **D** respectively. Working through the official sample questions (Mirantis publishes ~10) is high-yield.

## Where to go next

You've taken the DCA path. The natural follow-ons:

### Kubernetes

After Docker comes Kubernetes — it orchestrates the same images, with a richer model and a bigger ecosystem. The mapping from notebook 09 makes the transition concrete: Service -> Deployment, Task -> Pod, Stack -> Helm chart, etc.

- **CKAD** — Certified Kubernetes Application Developer. App-level focus (Pods, Deployments, Services, ConfigMaps, Secrets). The right next cert after DCA.
- **CKA** — Certified Kubernetes Administrator. Cluster-level focus (kubeadm, etcd, networking, troubleshooting nodes). Take after CKAD.
- **CKS** — Certified Kubernetes Security Specialist. Security focus on top of CKA.

### Cloud-specific containers

If your work runs in a cloud, the cloud-vendor certs make the platform-specific glue concrete:

- AWS: ECS, EKS, App Runner, Fargate. Bundled in the AWS Certified DevOps or Solutions Architect certs.
- GCP: Cloud Run, GKE. Bundled in the Google Cloud Professional Cloud Architect.
- Azure: AKS, Container Apps. Bundled in AZ-400 (Azure DevOps Engineer).

### Adjacent depth

- **CI/CD with containers** — GitHub Actions, GitLab CI, ArgoCD, Flux. Where the images you build get to production.
- **Observability** — Prometheus, Grafana, Loki, OpenTelemetry. What you bolt onto containerised services to see what they're doing in prod.
- **Service meshes** — Istio, Linkerd. Layer-7 routing, retries, mTLS between services.
- **Supply chain** — Cosign, in-toto, SLSA, OPA/Kyverno. The notebook 08 supply-chain section, taken to production.

The path from "I've never touched a container" (notebook 01) to "I can architect a containerised production system with supply-chain attestations, multi-cluster failover, and a service mesh" is real, and you've taken the first half of it in this curriculum.

Go book the exam.